Carlos Emiliano Mendoza Hernández

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import t
from scipy.stats import ttest_ind

# Análisis de ideología política. 

A través de datos del General Social Survey analizarás el cambio en la orientación política de la población estadounidense, para los años 1974 y 2018. Se tienen dos variables de interés: POLVIEWS y PARTYID. La primera representa en una escala del 1 al 7 la orientación política del/la entrevistad@, siendo 1 extremadamente liberal y 7 extremadamente conservador. Por otro lado, la segunda variable representa la afiliación partidista, siendo 0 fuertemente demócrata y 6 fuertemente republicano. Analiza la media de orientación política para los casos fuertemente demócrata y fuertemente republicano. Para estudiar el cambio en el tiempo de la diferencia de estas medias calcula dos intervalos de confianza para $\mu_6-\mu_0$. Asimismo, realiza pruebas de hipótesis para la pregunta ¿Hay una diferencia entre la orientación política media de los fuertemente demócratas y fuertemente republicanos?. Qué consideras que es más informativo ¿Un intervalo de confianza o una prueba de hipótesis? Comenta lo encontrado.

Los datos los puedes encontrar aquí: https://sda.berkeley.edu/sdaweb/analysis/?dataset=gss18
Puedes pre-visualizar alguna variable en la sección variable selection. Para descargar los datos que usarás está la pestaña "Download custom subset", donde podrás seleccionar el tipo de archivo, las observaciones/los casos de interés y las variables/features que necesitas. Recuerda incluir "YEAR" en las variables.

In [2]:
# Función para el intervalo de confianza de una diferencia de medias, suponiendo distr. pivotal t de student
def interv_conf_dosmedias(y1, y2, m_var = True, alfa = 0.05):
    n1 = len(y1); n2 = len(y2)
    v1 = np.var(y1, ddof = 1); v2 = np.var(y2, ddof = 1)
    if m_var:
        df = n1+n2-2
        var_resta = (((n1-1)*v1+(n2-1)*v2)/(n1+n2-2))*(1/n1+1/n2)
    else:
        df = (v1/n1+v2/n2)**2/(v1**2/(n1**2*(n1-1))+v2**2/(n2**2*(n2-1)))
        var_resta = v1/n1+v2/n2
    se = np.sqrt(var_resta)
    t_a = t.isf(alfa/2,df)
    resta_medias = np.mean(y1)-np.mean(y2)
    interv_conf = resta_medias+np.array([-1, 1])*t_a*se
    confianza = 1-alfa
    return resta_medias, interv_conf, confianza, df

In [3]:
df = pd.read_csv('./data/sub-data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3832 entries, 0 to 3831
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   POLVIEWS  3832 non-null   int64
 1   PARTYID   3832 non-null   int64
 2   YEAR      3832 non-null   int64
dtypes: int64(3)
memory usage: 89.9 KB


In [4]:
df = df[~df['POLVIEWS'].isin([8, 9])]
df['POLVIEWS'].unique()

array([4, 5, 6, 2, 7, 3, 1])

In [5]:
df

,POLVIEWS,PARTYID,YEAR
0,4,2,1974
1,5,0,1974
2,6,0,1974
3,6,0,1974
4,6,1,1974
...,...,...,...
3826,4,5,2018
3827,4,3,2018
3828,5,5,2018
3829,4,3,2018


## Filtrar datos

In [6]:
# 1974:
df_1974 = df[df["YEAR"] == 1974]
dems_1974 = df_1974[df_1974["PARTYID"] == 0]["POLVIEWS"]
reps_1974 = df_1974[df_1974["PARTYID"] == 6]["POLVIEWS"]

# 2018:
df_2018 = df[df["YEAR"] == 2018]
dems_2018 = df_2018[df_2018["PARTYID"] == 0]["POLVIEWS"]
reps_2018 = df_2018[df_2018["PARTYID"] == 6]["POLVIEWS"]

## Calcular intervalos de confianza y pruebas de hipótesis

In [7]:
resta_medias_1974, ic_1974, conf_1974, df_1974 = interv_conf_dosmedias(reps_1974, dems_1974, alfa=0.05)
resta_medias_2018, ic_2018, conf_2018, df_2018 = interv_conf_dosmedias(reps_2018, dems_2018, alfa=0.05)

t_1974, p_1974 = ttest_ind(reps_1974, dems_1974, equal_var=True)
t_2018, p_2018 = ttest_ind(reps_2018, dems_2018, equal_var=True)

## Resultados

In [ ]:
print("----- 1974 -----")
print(f"Diferencia de medias (reps - dems): {resta_medias_1974:.3f}")
print(f"Intervalo de confianza 95%: ({ic_1974[0]:.3f}, {ic_1974[1]:.3f}) con {df_1974} grados de libertad")
print(f"t: {t_1974:.3f}, p-valor bilateral: {p_1974}")
print("\n----- 2018 -----")
print(f"Diferencia de medias (reps - dems): {resta_medias_2018:.3f}")
print(f"Intervalo de confianza 95%: ({ic_2018[0]:.3f}, {ic_2018[1]:.3f}) con {df_2018} grados de libertad")
print(f"t: {t_2018:.3f}, p-valor bilateral: {p_2018}")

----- 1974 -----
Diferencia de medias (reps - dems): 0.941
Intervalo de confianza 95%: (0.618, 1.264) con 330 grados de libertad
t: 5.729, p-valor bilateral: 2.2741011426597366e-08

----- 2018 -----
Diferencia de medias (reps - dems): 2.742
Intervalo de confianza 95%: (2.529, 2.955) con 618 grados de libertad
t: 25.236, p-valor bilateral: 4.009417938864369e-97


## Interpretación de resultados

## Pruebas de hipótesis
**1974**

 * $t = 5.729$, p-valor $< 0.05$.

Con un p-valor  pequeño, rechazamos $H_{0}: \mu_{6} = \mu_{0}$ al nivel $\alpha = 0.05$. Esto nos dice que en 1974 hubo evidencia estadística muy fuerte para afirmar que la orientación política media de los republicanos fuertes ($\mu_{6}$) era distinta (y, de hecho, mayor) que la de los demócratas fuertes ($\mu_{0}$).

**2018**

 * $t = 25.236$, p-valor bilateral $< 0.05$.

De nuevo, el p-valor prácticamente cero nos lleva a rechazar $H_{0}$. Por lo tanto, en 2018 también existe evidencia estadística de que $\mu_{6} \neq \mu_{0}$. En ambos años la prueba de hipótesis confirma que no es razonable pensar que las medias sean iguales entre demócratas fuertes y republicanos fuertes.

## Intervalos de confianza

**1974**

$$ \mathrm{IC}_{95\%}\bigl(\mu_{6}-\mu_{0}\bigr) \;=\; (0.618,\;1.264), \quad \text{con } 330 \text{ grados de libertad}.$$

El intervalo no contiene el 0, lo que coincide con el p-valor pequeño. Esto indica que, con 95 % de confianza, la verdadera brecha poblacional en 1974 estaba entre 0.618 y 1.264 unidades (en la escala 1–7).

**2018**:

$$
   \mathrm{IC}_{95\%}\bigl(\mu_{6}-\mu_{0}\bigr) \;=\; (2.529,\;2.955),
   \quad \text{con } 618 \text{ grados de libertad}.
$$
Tampoco contiene el 0. Esto significa que, con 95 % de confianza, la verdadera diferencia en 2018 estaba entre 2.529 y 2.955 unidades.

## ¿Qué aporta más el IC o la prueba de hipótesis?

La prueba de hipótesis nos dice si existe o no evidencia suficiente para rechazar la igualdad de medias. En ambos años, el p-valor casi cero nos da la certeza de que sí existe una diferencia significativa de medias entre los dos grupos.
Sin embargo, el intervalo de confianza agrega información importante como:

  1. En 1974, la brecha es de alrededor de 1 unidad; en 2018, cerca de 2.7 unidades.
  2. El ancho del IC (por ejemplo, para 2018 es de aproximadamente 0.426) muestra que tan “confiables” son esos valores.
  3. El intervalo completamente por encima de 0 confirma que los republicanos fuertes se ubican más conservadores, y permite ver en cuánto exactamente.

La prueba de hipótesis (con p-valor) nos dice si hay diferencia real en la población, mientras que el IC nos dice cuánto difieren realmente y con qué precisión lo estimamos. Por ello, el intervalo de confianza es más informativo.

En este caso, si solo hubiéramos mirado la prueba de hipótesis, sabríamos que hay diferencia, pero no cuánto ha cambiado desde 1974 a 2018. Gracias a los IC, vemos claramente que la brecha promedio casi se triplico.

## Conclusiones
La diferencia de medias entre demócratas y republicanos fuertes ha aumentado significativamente entre 1974 y 2018. En 1974, la diferencia era de aproximadamente 1 unidad, mientras que en 2018 fue de alrededor de 2.7 unidades. Esto sugiere un aumento en la polarización política en Estados Unidos durante este período.